In [3]:
# Optimizing Indian Monetary Policy Using Metaheuristic Algorithms
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from math import pi
import random
import math
import time
import warnings

warnings.filterwarnings("ignore")

class DataAnalysis:
    @staticmethod
    def load_data(filepath: str) -> pd.DataFrame:
        df = pd.read_excel(filepath)
        return df

    @staticmethod
    def analyze_correlations(df: pd.DataFrame) -> list:
        gdp_correlations = {}
        for col in df.columns:
            if col not in ("Year - Month", "Interpolated GDP"):
                try:
                    valid = df[col].notna()
                    corr, _ = pearsonr(df.loc[valid, col],
                                       df.loc[valid, "Interpolated GDP"])
                    gdp_correlations[col] = corr
                except Exception:
                    pass
        return sorted(gdp_correlations.items(),
                       key=lambda x: abs(x[1]), reverse=True)

class EconomicOptimizer:
    """
    Defines the multi-objective fitness function for monetary policy.

    Decision variables : [Policy Repo Rate, CRR, SLR]
    Objective          : Maximise weighted economic welfare score
                         (60 % GDP score + 30 % inflation control + 10 % liquidity)
    Constraints        : 4.0 ≤ repo ≤ 8.5, 3.0 ≤ CRR ≤ 6.0, 18.0 ≤ SLR ≤ 24.0
    """

    # Search-space bounds  [repo_rate_low, crr_low, slr_low]
    LOWER = np.array([4.0, 3.0, 18.0])
    UPPER = np.array([8.5, 6.0, 24.0])

    def __init__(self, df: pd.DataFrame):
        self.df = df
        # Historical statistics (used only for reference; not in objective)
        self.gdp_mean = df["Interpolated GDP"].mean()
        self.gdp_std  = df["Interpolated GDP"].std()
        self.inflation_mean = df["CPI inflation (Base = 2012)"].mean()
        self.inflation_std  = df["CPI inflation (Base = 2012)"].std()

        # Empirical policy sensitivity coefficients
        self._policy_rate_gdp_coeff = 0.15
        self._crr_gdp_coeff         = 0.12
        self._slr_gdp_coeff         = 0.10

    def objective_function(self, solution: np.ndarray) -> float:
        policy_rate, crr, slr = float(solution[0]), float(solution[1]), float(solution[2])

        if not self.check_constraints(solution):
            return -1e10

        # GDP component (higher repo / CRR / SLR mildly suppresses growth)
        gdp_score = 100 - (
            policy_rate * self._policy_rate_gdp_coeff
            + crr        * self._crr_gdp_coeff
            + slr        * self._slr_gdp_coeff
        )

        # Inflation control: Gaussian penalty centred at 6 % repo rate
        inflation_control = 100 * np.exp(-((policy_rate - 6.0) ** 2) / 2)

        # Liquidity management: penalise deviations from CRR=4.5, SLR=21
        liquidity_score = 100 - abs((crr - 4.5) * 5 + (slr - 21) * 2)

        # Weighted composite (weights sum to 1)
        fitness = (0.60 * gdp_score
                   + 0.30 * inflation_control
                   + 0.10 * liquidity_score)
        return float(fitness)

    def check_constraints(self, solution: np.ndarray) -> bool:
        """Return True when all decision variables are within bounds."""
        policy_rate, crr, slr = solution[0], solution[1], solution[2]
        return (self.LOWER[0] <= policy_rate <= self.UPPER[0]
                and self.LOWER[1] <= crr       <= self.UPPER[1]
                and self.LOWER[2] <= slr       <= self.UPPER[2])

#GA
class GeneticAlgorithm:
    def __init__(self, optimizer: EconomicOptimizer,
                 population_size: int = 50,
                 generations: int = 100,
                 mutation_rate: float = 0.15,
                 crossover_rate: float = 0.8,
                 seed: int = 42):
        self.optimizer       = optimizer
        self.population_size = population_size
        self.generations     = generations
        self.mutation_rate   = mutation_rate
        self.crossover_rate  = crossover_rate
        self.seed            = seed

        self.best_fitness_history: list = []
        self.best_solution: np.ndarray  = None
        self.best_fitness: float        = -np.inf

    def _initialize_population(self) -> list:
        lb, ub = self.optimizer.LOWER, self.optimizer.UPPER
        return [np.random.uniform(lb, ub) for _ in range(self.population_size)]

    def _evaluate(self, population: list) -> list:
        return [self.optimizer.objective_function(ind) for ind in population]

    def _tournament(self, population: list, fitness: list,
                    k: int = 3) -> np.ndarray:
        idx = np.random.choice(len(population), k, replace=False)
        best = idx[np.argmax([fitness[i] for i in idx])]
        return population[best].copy()

    def _crossover(self, p1: np.ndarray, p2: np.ndarray) -> np.ndarray:
        if np.random.random() < self.crossover_rate:
            pt = np.random.randint(1, 3)
            return np.concatenate([p1[:pt], p2[pt:]])
        return p1.copy()

    def _mutate(self, ind: np.ndarray) -> np.ndarray:
        child = ind.copy()
        if np.random.random() < self.mutation_rate:
            idx = np.random.randint(0, 3)
            sigmas = [0.5, 0.3, 1.0]
            child[idx] += np.random.normal(0, sigmas[idx])
            child[idx] = np.clip(child[idx],
                                  self.optimizer.LOWER[idx],
                                  self.optimizer.UPPER[idx])
        return child


    def run(self):
        """Execute GA. Returns (best_solution, best_fitness, history)."""
        np.random.seed(self.seed)
        pop = self._initialize_population()

        for _ in range(self.generations):
            fit = self._evaluate(pop)
            best_idx = int(np.argmax(fit))

            if fit[best_idx] > self.best_fitness:
                self.best_fitness = fit[best_idx]
                self.best_solution = pop[best_idx].copy()

            self.best_fitness_history.append(self.best_fitness)

            new_pop = []
            for _ in range(self.population_size):
                p1 = self._tournament(pop, fit)
                p2 = self._tournament(pop, fit)
                child = self._crossover(p1, p2)
                child = self._mutate(child)
                new_pop.append(child)
            pop = new_pop

        return self.best_solution, self.best_fitness, self.best_fitness_history


#PSO
class ParticleSwarmOptimization:
    """Standard PSO (inertia-weight variant)."""

    def __init__(self, optimizer: EconomicOptimizer,
                 num_particles: int = 40,
                 iterations: int = 100,
                 w: float = 0.7,
                 c1: float = 1.5,
                 c2: float = 1.5,
                 seed: int = 42):
        self.optimizer     = optimizer
        self.num_particles = num_particles
        self.iterations    = iterations
        self.w, self.c1, self.c2 = w, c1, c2
        self.seed          = seed

        self.best_fitness_history: list = []
        self.global_best: np.ndarray    = None
        self.global_best_fitness: float = -np.inf

    def _init_swarm(self):
        lb, ub = self.optimizer.LOWER, self.optimizer.UPPER
        pos = np.random.uniform(lb, ub, (self.num_particles, 3))
        vel = np.random.uniform(-1, 1,  (self.num_particles, 3))
        return pos, vel

    def _clip(self, pos: np.ndarray) -> np.ndarray:
        return np.clip(pos, self.optimizer.LOWER, self.optimizer.UPPER)

    def run(self):
        """Execute PSO. Returns (best_solution, best_fitness, history)."""
        np.random.seed(self.seed)
        pos, vel = self._init_swarm()

        pbest     = pos.copy()
        pbest_fit = np.array([self.optimizer.objective_function(p) for p in pos])

        best_idx              = int(np.argmax(pbest_fit))
        self.global_best      = pbest[best_idx].copy()
        self.global_best_fitness = pbest_fit[best_idx]

        for _ in range(self.iterations):
            for i in range(self.num_particles):
                r1, r2 = np.random.random(3), np.random.random(3)
                vel[i] = (self.w  * vel[i]
                          + self.c1 * r1 * (pbest[i]           - pos[i])
                          + self.c2 * r2 * (self.global_best   - pos[i]))
                vel[i] = np.clip(vel[i], -2, 2)
                pos[i] = self._clip(pos[i] + vel[i])

                fit = self.optimizer.objective_function(pos[i])
                if fit > pbest_fit[i]:
                    pbest_fit[i] = fit
                    pbest[i]     = pos[i].copy()
                if fit > self.global_best_fitness:
                    self.global_best_fitness = fit
                    self.global_best         = pos[i].copy()

            self.best_fitness_history.append(self.global_best_fitness)

        return self.global_best, self.global_best_fitness, self.best_fitness_history

#SA
class SimulatedAnnealing:
    def __init__(self, optimizer: EconomicOptimizer,
                 initial_temp: float = 100.0,
                 cooling_rate: float = 0.95,
                 iterations_per_temp: int = 50,
                 min_temp: float = 0.01,
                 seed: int = 42):
        self.optimizer           = optimizer
        self.initial_temp        = initial_temp
        self.cooling_rate        = cooling_rate
        self.iterations_per_temp = iterations_per_temp
        self.min_temp            = min_temp
        self.seed                = seed

        self.best_fitness_history: list = []
        self.best_solution: np.ndarray  = None
        self.best_fitness: float        = -np.inf

    def _neighbor(self, sol: np.ndarray) -> np.ndarray:
        nb  = sol.copy()
        idx = np.random.randint(0, 3)
        sigmas = [0.3, 0.2, 0.5]
        nb[idx] += np.random.normal(0, sigmas[idx])
        nb[idx] = np.clip(nb[idx],
                           self.optimizer.LOWER[idx],
                           self.optimizer.UPPER[idx])
        return nb

    @staticmethod
    def _accept(curr_fit: float, new_fit: float, temp: float) -> bool:
        if new_fit > curr_fit:
            return True
        return np.random.random() < np.exp((new_fit - curr_fit) / temp)

    def run(self):
        """Execute SA. Returns (best_solution, best_fitness, history)."""
        np.random.seed(self.seed)
        lb, ub = self.optimizer.LOWER, self.optimizer.UPPER

        curr     = np.random.uniform(lb, ub)
        curr_fit = self.optimizer.objective_function(curr)

        self.best_solution = curr.copy()
        self.best_fitness  = curr_fit
        temp               = self.initial_temp

        while temp > self.min_temp:
            for _ in range(self.iterations_per_temp):
                nb     = self._neighbor(curr)
                nb_fit = self.optimizer.objective_function(nb)

                if self._accept(curr_fit, nb_fit, temp):
                    curr, curr_fit = nb.copy(), nb_fit

                if curr_fit > self.best_fitness:
                    self.best_fitness  = curr_fit
                    self.best_solution = curr.copy()

            self.best_fitness_history.append(self.best_fitness)
            temp *= self.cooling_rate

        return self.best_solution, self.best_fitness, self.best_fitness_history


#ACO
class AntColonyOptimization:
    def __init__(self, optimizer: EconomicOptimizer,
                 num_ants: int = 30,
                 iterations: int = 100,
                 evaporation_rate: float = 0.1,
                 q: float = 10.0,
                 num_bins: int = 10,
                 seed: int = 42):
        self.optimizer        = optimizer
        self.num_ants         = num_ants
        self.iterations       = iterations
        self.evaporation_rate = evaporation_rate
        self.q                = q
        self.num_bins         = num_bins
        self.seed             = seed

        self.pheromone = {
            "policy_rate": np.ones(num_bins) * 0.5,
            "crr":         np.ones(num_bins) * 0.5,
            "slr":         np.ones(num_bins) * 0.5,
        }

        self.best_fitness_history: list = []
        self.best_solution: np.ndarray  = None
        self.best_fitness: float        = -np.inf

    def _bin_to_val(self, b: int, lo: float, hi: float) -> float:
        return lo + (b / (self.num_bins - 1)) * (hi - lo)

    def _val_to_bin(self, v: float, lo: float, hi: float) -> int:
        return int(np.clip(round((v - lo) / (hi - lo) * (self.num_bins - 1)),
                           0, self.num_bins - 1))

    def _construct(self):
        lb, ub = self.optimizer.LOWER, self.optimizer.UPPER
        keys   = ["policy_rate", "crr", "slr"]
        bins, vals = [], []
        for k, lo, hi in zip(keys, lb, ub):
            ph  = self.pheromone[k]
            pr  = ph / ph.sum()
            b   = np.random.choice(self.num_bins, p=pr)
            bins.append(b)
            vals.append(self._bin_to_val(b, lo, hi))
        return np.array(vals), bins

    def _update_pheromone(self, solutions, fitnesses):
        lb, ub = self.optimizer.LOWER, self.optimizer.UPPER
        keys   = ["policy_rate", "crr", "slr"]

        # Evaporate
        for k in keys:
            self.pheromone[k] *= (1 - self.evaporation_rate)

        # Deposit
        for sol, fit in zip(solutions, fitnesses):
            deposit = self.q / max(1.0, (100.0 - fit + 1e-9))
            for dim, (k, lo, hi) in enumerate(zip(keys, lb, ub)):
                b = self._val_to_bin(sol[dim], lo, hi)
                self.pheromone[k][b] += deposit

    def run(self):
        """Execute ACO. Returns (best_solution, best_fitness, history)."""
        np.random.seed(self.seed)

        for _ in range(self.iterations):
            solutions, fitnesses = [], []

            for _ in range(self.num_ants):
                sol, _ = self._construct()
                fit    = self.optimizer.objective_function(sol)
                solutions.append(sol)
                fitnesses.append(fit)

                if fit > self.best_fitness:
                    self.best_fitness  = fit
                    self.best_solution = sol.copy()

            self._update_pheromone(solutions, fitnesses)
            self.best_fitness_history.append(self.best_fitness)

        return self.best_solution, self.best_fitness, self.best_fitness_history

#Visualisation
def plot_results(results: dict, save_dir: str = ".") -> None:
    plt.style.use("seaborn-v0_8-darkgrid")
    algos   = list(results.keys())
    fits    = [results[a]["fitness"]      for a in algos]
    times   = [results[a]["time"]         for a in algos]
    pr_vals = [results[a]["solution"][0]  for a in algos]
    crr_vals= [results[a]["solution"][1]  for a in algos]
    slr_vals= [results[a]["solution"][2]  for a in algos]
    colors  = ["#2ecc71", "#3498db", "#e74c3c", "#f39c12"]

    # 3. Speed vs quality
    fig, ax = plt.subplots(figsize=(10, 7))
    for algo, t, f, clr in zip(algos, times, fits, colors):
        ax.scatter(t, f, s=300, c=clr, edgecolors="black", linewidth=2, zorder=3)
        ax.annotate(algo, (t, f), xytext=(5, 5),
                    textcoords="offset points", fontsize=11, fontweight="bold")
    z = np.polyfit(times, fits, 1)
    xt = np.linspace(min(times) - 0.05, max(times) + 0.05, 100)
    ax.plot(xt, np.poly1d(z)(xt), "r--", alpha=0.5, linewidth=2, label="Trend")
    # Star on best efficiency (highest fitness / unit time)
    eff  = [f / t for f, t in zip(fits, times)]
    best = algos[int(np.argmax(eff))]
    bi   = algos.index(best)
    ax.scatter(times[bi], fits[bi], s=600, marker="*", c="gold",
               edgecolors="red", linewidth=2, label=f"⭐ Best efficiency ({best})", zorder=5)
    ax.set_xlabel("Execution Time (s)", fontweight="bold")
    ax.set_ylabel("Fitness Score", fontweight="bold")
    ax.set_title("Speed vs Quality Trade-off", fontweight="bold", fontsize=14, pad=15)
    ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{save_dir}/speed_vs_quality.png", dpi=300, bbox_inches="tight")
    plt.close()

def main(filepath: str = "final_data.xlsx") -> None:
    print("  OPTIMIZING INDIAN MONETARY POLICY USING METAHEURISTIC ALGORITHMS")

    df = DataAnalysis.load_data(filepath)
    print(f"  ✓ {df.shape[0]} rows × {df.shape[1]} columns loaded.")

    # --- correlations ---
    print("\nTop correlations with GDP:")
    top_corrs = DataAnalysis.analyze_correlations(df)[:5]
    for col, r in top_corrs:
        print(f"  {col:45s} r = {r:+.4f}")

    # --- optimizer ---
    optimizer = EconomicOptimizer(df)

    # --- run four algorithms ---
    print("  RUNNING OPTIMIZATION ALGORITHMS")

    algorithms = {
        "GA" : GeneticAlgorithm(optimizer,  population_size=50,  generations=100,   seed=42),
        "PSO": ParticleSwarmOptimization(optimizer, num_particles=40, iterations=100, seed=42),
        "SA" : SimulatedAnnealing(optimizer, initial_temp=100,    cooling_rate=0.95,  seed=42),
        "ACO": AntColonyOptimization(optimizer, num_ants=30,      iterations=100,     seed=42),
    }

    results: dict = {}
    labels = {"GA": "Genetic Algorithm",
               "PSO": "Particle Swarm Optimization",
               "SA": "Simulated Annealing",
               "ACO": "Ant Colony Optimization"}

    for i, (tag, algo) in enumerate(algorithms.items(), 1):
        print(f"\n[{i}/4] {labels[tag]} …")
        t0 = time.perf_counter()
        sol, fit, hist = algo.run()
        elapsed = time.perf_counter() - t0
        results[tag] = {"solution": sol, "fitness": fit,
                        "history": hist, "time": elapsed}
        print(f"  Best Fitness : {fit:.4f}")
        print(f"  Repo Rate    : {sol[0]:.4f} %")
        print(f"  CRR          : {sol[1]:.4f} %")
        print(f"  SLR          : {sol[2]:.4f} %")
        print(f"  Time         : {elapsed:.4f} s")

    # --- summary table ---
    print("\n" + "=" * 68)
    print("  RESULTS SUMMARY")
    print("=" * 68)
    ranked = sorted(results.items(), key=lambda x: x[1]["fitness"], reverse=True)
    print(f"\n{'Rank':<6}{'Algorithm':<6}{'Fitness':>10}  {'Repo%':>7}  "
          f"{'CRR%':>7}  {'SLR%':>7}  {'Time(s)':>9}")
    print("-" * 60)
    for rank, (tag, res) in enumerate(ranked, 1):
        s = res["solution"]
        print(f"  {rank}    {tag:<6}  {res['fitness']:>8.4f}  "
              f"{s[0]:>7.4f}  {s[1]:>7.4f}  {s[2]:>7.4f}  "
              f"{res['time']:>9.4f}")

    winner = ranked[0][0]
    ws = ranked[0][1]["solution"]
    print(f"\n  Winner : {labels[winner]}")
    print(f"     Optimal Repo Rate : {ws[0]:.4f} %")
    print(f"     Optimal CRR       : {ws[1]:.4f} %")
    print(f"     Optimal SLR       : {ws[2]:.4f} %")

    # --- multi-seed validation ---
    print("\n" + "=" * 68)
    print("  MULTI-SEED VALIDATION  (PSO, 5 seeds)")
    print("=" * 68)
    seed_results = []
    for s in [0, 7, 13, 21, 42]:
        pso_val = ParticleSwarmOptimization(optimizer, seed=s)
        sol_v, fit_v, _ = pso_val.run()
        seed_results.append((s, fit_v, sol_v))
        print(f"  seed={s:2d}  fit={fit_v:.4f}  "
              f"repo={sol_v[0]:.4f}  crr={sol_v[1]:.4f}  slr={sol_v[2]:.4f}")

    avg_sol = np.mean([r[2] for r in seed_results], axis=0)
    std_fit = np.std( [r[1] for r in seed_results])
    print(f"\n  Mean Repo Rate : {avg_sol[0]:.4f} %  |  Fitness std : {std_fit:.6f}")

    plot_results(results)


if __name__ == "__main__":
    main("final_data.xlsx")

  OPTIMIZING INDIAN MONETARY POLICY USING METAHEURISTIC ALGORITHMS
  ✓ 184 rows × 18 columns loaded.

Top correlations with GDP:
  Exchange Rate of Indian Rupee (month end)     r = +0.8922
  Statutory Liquidity Ratio                     r = -0.8503
  Reverse Repo Rate                             r = -0.7858
  M3 (in crore)                                 r = +0.7283
  Consumer Price Index for Industrial Workers   r = -0.6272
  RUNNING OPTIMIZATION ALGORITHMS

[1/4] Genetic Algorithm …
  Best Fitness : 97.9542
  Repo Rate    : 5.9981 %
  CRR          : 5.5010 %
  SLR          : 18.4973 %
  Time         : 0.6370 s

[2/4] Particle Swarm Optimization …
  Best Fitness : 97.9697
  Repo Rate    : 5.9970 %
  CRR          : 5.7000 %
  SLR          : 18.0000 %
  Time         : 0.2191 s

[3/4] Simulated Annealing …
  Best Fitness : 97.9510
  Repo Rate    : 6.0152 %
  CRR          : 5.6491 %
  SLR          : 18.0572 %
  Time         : 0.4039 s

[4/4] Ant Colony Optimization …
  Best Fitness : 97.9